# ZINC Streaming Demo

This notebook showcases the schema-frozen streaming training path for `ConditionalNodeFieldGraphGenerator`.

- source: raw ZINC CSV
- warmup: first 1000 accepted graphs
- stream limit: `0.1`
- targets: none
- final cells: generate 7 graphs without feasibility filtering, then 7 with filtering


In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import random

import numpy as np
from IPython.core.display import HTML

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

from conditional_node_field_graph_generator.notebooks import configure_notebook
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from abstractgraph_graphicalizer.chem import download_zinc_dataset, draw_molecules
from conditional_node_field_graph_generator.extensions.demo import show_molecules
from conditional_node_field_graph_generator.extensions.demo.pipeline import build_graph_generator


PyTorch version: 2.2.2
CUDA available: False
Enabling RDKit 2025.09.3 jupyter extensions


/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
RANDOM_SEED = 7
STREAM_LIMIT = 0.05
WARMUP_SIZE = 2048
STREAM_BATCH_SIZE = 256
MAXIMUM_EPOCHS = 5
EMBEDDING_DIM = 64
MODEL_NAME = f'zinc-streaming-n{EMBEDDING_DIM}-s{STREAM_LIMIT}-w{WARMUP_SIZE}-b{STREAM_BATCH_SIZE}-e{MAXIMUM_EPOCHS}'
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
DECODER_N_JOBS = -1

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [3]:
csv_path = download_zinc_dataset(ZINC_DATA_ROOT)
print(f'ZINC CSV: {csv_path}')

graph_generator = build_graph_generator(
    latent_embedding_dimension=EMBEDDING_DIM,
    number_of_transformer_layers=2,
    transformer_attention_head_count=4,
    maximum_epochs=MAXIMUM_EPOCHS,
    batch_size=STREAM_BATCH_SIZE,
    verbose=1,
    decoder_n_jobs=DECODER_N_JOBS,
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
)
graph_generator.graph_decoder.diagnostic_graph_renderer = draw_molecules


ZINC CSV: /Users/fabriziocosta/Resilio Sync/Sync/Projects/NodeField/notebooks/datasets/zinc/zinc_250k.csv
Configured graph generator model_name=zinc-streaming-n64-s0.05-w2048-b256-e5 model_dir=/Users/fabriziocosta/Resilio Sync/Sync/Projects/NodeField/.artifacts/saved_generators


In [ ]:
graph_generator.fit_from_stream(
    csv_path,
    'zinc_csv',
    warmup_size=WARMUP_SIZE,
    batch_size=STREAM_BATCH_SIZE,
    limit=STREAM_LIMIT,
    random_state=RANDOM_SEED,
    verbose=True,
)

print('stream_seen_ =', graph_generator.stream_seen_)
print('stream_warmup_count_ =', graph_generator.stream_warmup_count_)
print('stream_training_seen_ =', graph_generator.stream_training_seen_)
print('stream_training_accepted_ =', graph_generator.stream_training_accepted_)
print('stream_training_skipped_ =', graph_generator.stream_training_skipped_)
print('stream_acceptance_rate_ =', graph_generator.stream_acceptance_rate_)


Warmup fitting on 2048 streamed graphs.
Fitting feasibility estimator on 2048 graphs
Supervision plan:
  node_labels: mode=learned, enabled. 9 node labels detected.
  edge_labels: mode=learned, enabled. 4 edge labels detected.
  direct_edges: mode=learned, enabled, horizon=1. Generator should learn horizon-1 edge presence for the decoder.
  auxiliary_locality: mode=disabled, disabled. No auxiliary locality is needed when locality_horizon=1.
adj_mtx_to_targets[direct_edge, horizon=1]: sampling 306185 pairs (50.00%) from 612370 total pairs (pos=204124, neg=408246, negative_sample_factor=1, sampling_strategy=stratified_preserve, target_positive_ratio=0.500).
adj_mtx_to_targets[direct_edge, horizon=1]: using pos=102062, neg=204123, positive_ratio=0.333.
Warmup schema frozen with up to 36 nodes per graph.
Lambda settings: degree=2.000, node_exist=2.000, node_count=0.500, node_label=2.000, edge_label=2.000, direct_edge=2.000, edge_count=0.500, deg_edge_consistency=0.500, aux_edge=1.000
Direc

GPU available: False, used: False
TPU available: False, using: 0 TPU cores


train batch    1: accepted=    512 skipped=      0 | total=  154.1044 node_field=   24.9940 deg=  1.6449
train batch    2: accepted=    768 skipped=      0 | total=  155.4838 node_field=   24.9922 deg=  1.6049
train batch    3: accepted=   1024 skipped=      0 | total=  156.5944 node_field=   24.9876 deg=  1.5849
train batch    4: accepted=   1280 skipped=      0 | total=  153.3521 node_field=   24.9965 deg=  1.5442
train batch    5: accepted=   1536 skipped=      0 | total=  148.9953 node_field=   24.9971 deg=  1.5122
train batch    6: accepted=   1792 skipped=      0 | total=  146.7232 node_field=   25.0024 deg=  1.4949
train batch    7: accepted=   2048 skipped=      0 | total=  142.1787 node_field=   24.9904 deg=  1.4582
train batch    8: accepted=   2304 skipped=      0 | total=  143.5102 node_field=   25.0047 deg=  1.4319
train batch    9: accepted=   2560 skipped=      0 | total=  138.8254 node_field=   24.9965 deg=  1.4200
train batch   10: accepted=   2816 skipped=      1 | to

In [ ]:
raw_samples = graph_generator.sample(
    n_samples=7,
    apply_feasibility_filtering=False,
)
show_molecules(raw_samples, n=7, title='Streaming ZINC samples without feasibility filtering')


In [ ]:
if graph_generator.feasibility_estimator is None:
    raise RuntimeError('Feasibility estimator is unavailable in this environment.')

filtered_samples = graph_generator.sample(
    n_samples=7,
    apply_feasibility_filtering=True,
)
show_molecules(filtered_samples, n=7, title='Streaming ZINC samples with feasibility filtering')
